In [3]:
import joblib
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
pipeline = joblib.load(ROOT / "data" / "05_modelado" / "pipeline_preprocesamiento.pkl")

print(f"Tipo: {type(pipeline).__name__}")
print(f"\nTransformers:")
for name, transformer, cols in pipeline.transformers_:
    print(f"  {name}: {type(transformer).__name__} → columnas: {cols}")
    if hasattr(transformer, 'mean_'):
        for col, mean, std in zip(cols, transformer.mean_, transformer.scale_):
            print(f"    {col}: mean={mean:.3f}, std={std:.3f}")

# Ver qué columnas van a cada transformador
print(f"\nfeature_names_in_: {list(pipeline.feature_names_in_)}")

Tipo: ColumnTransformer

Transformers:
  num: Pipeline → columnas: ['cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso', 'rama', 'sexo', 'edad_entrada', 'pais_nombre', 'provincia', 'via_acceso', 'orden_preferencia', 'universidad_origen', 'tuvo_beca', 'n_anios_beca', 'situacion_laboral', 'nota_selectividad', 'max_pagos', 'indicador_interrupcion', 'anios_gap', 'anios_sin_beca']

feature_names_in_: ['cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso', 'rama', 'sexo', 'edad_entrada', 'pais_nombre', 'provincia', 'via_acceso', 'orden_preferencia', 'universidad_origen', 'tuvo_beca', 'n_anios_beca', 'situacion_laboral', 'nota_selectividad', 'max_pagos', 'indicador_interrupcion', 'anios_gap', 'anios_sin_beca']


In [4]:
# Ver el pipeline interno del transformador 'num'
num_pipeline = pipeline.named_transformers_['num']
print(f"Steps del pipeline 'num':")
for name, step in num_pipeline.steps:
    print(f"  {name}: {type(step).__name__}")
    if hasattr(step, 'mean_'):
        feat_names = ['cred_superados_anio_1er', 'nota_1er_anio', 'nota_acceso',
                      'rama', 'sexo', 'edad_entrada', 'pais_nombre', 'provincia',
                      'via_acceso', 'orden_preferencia', 'universidad_origen',
                      'tuvo_beca', 'n_anios_beca', 'situacion_laboral',
                      'nota_selectividad', 'max_pagos', 'indicador_interrupcion',
                      'anios_gap', 'anios_sin_beca']
        for col, mean, std in zip(feat_names, step.mean_, step.scale_):
            if col in ['nota_acceso', 'nota_1er_anio', 'n_anios_beca',
                       'situacion_laboral', 'via_acceso', 'edad_entrada']:
                print(f"    {col}: mean={mean:.3f}, std={std:.3f}")

Steps del pipeline 'num':
  imputer: SimpleImputer
  scaler: StandardScaler
    nota_1er_anio: mean=6.834, std=0.856
    nota_acceso: mean=8.365, std=1.758
    edad_entrada: mean=20.805, std=5.315
    via_acceso: mean=8.363, std=3.042
    n_anios_beca: mean=1.730, std=1.578
    situacion_laboral: mean=7.215, std=3.675


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
df = pd.read_parquet(ROOT / "data" / "automl" / "dataset_final_tfm.parquet")

# ¿Qué tasa de abandono tienen los grupos por nota_acceso?
df['grupo_nota'] = pd.cut(df['nota_acceso'], 
                           bins=[0, 6, 8, 10, 12, 14],
                           labels=['<6', '6-8', '8-10', '10-12', '12-14'])
print(df.groupby('grupo_nota')['abandono'].agg(['mean', 'count']).round(3))


             mean  count
grupo_nota              
<6          0.524   3035
6-8         0.391  10949
8-10        0.223  11204
10-12       0.105   4478
12-14       0.076   1388


C:\Users\mjmor\AppData\Local\Temp\ipykernel_23584\1724100194.py:12: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('grupo_nota')['abandono'].agg(['mean', 'count']).round(3))


In [6]:
import joblib, pandas as pd
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
pipeline = joblib.load(ROOT / "data" / "05_modelado" / "pipeline_preprocesamiento.pkl")
modelo   = joblib.load(ROOT / "data" / "05_modelado" / "models" / "CatBoost__balanced.pkl")

# Simular perfil con nota 5 vs nota 12 — todo lo demás igual
perfil_base = {
    'nota_acceso':      5.0,
    'situacion_laboral': 11,   # No trabaja
    'n_anios_beca':     2,
    'edad_entrada':     19,
    'via_acceso':       10,    # Bachillerato
    'nota_1er_anio':    6.834, # media training
    'sexo':             0,
    'tuvo_beca':        1,
    'anios_sin_beca':   2,
    'nota_selectividad': 6.0,
    'orden_preferencia': 1,
    'anios_gap':        0,
    'universidad_origen': 40,
    'rama':             3,
    'pais_nombre':      1,
    'provincia':        1,
    'cred_superados_anio_1er': 40.0,
    'max_pagos':        1.0,
    'indicador_interrupcion': 0,
}

cols = list(pipeline.feature_names_in_)
for nota in [5.0, 8.0, 12.0, 14.0]:
    perfil_base['nota_acceso'] = nota
    fila = {col: perfil_base.get(col, 0) for col in cols}
    X = pd.DataFrame([fila], columns=cols)
    X_prep = pipeline.transform(X)
    prob = modelo.predict_proba(X_prep)[0, 1]
    print(f"nota_acceso={nota:.0f} → prob={prob*100:.1f}%")

nota_acceso=5 → prob=50.9%
nota_acceso=8 → prob=58.1%
nota_acceso=12 → prob=47.8%
nota_acceso=14 → prob=52.5%


In [7]:
perfil_base['nota_1er_anio'] = 0.0
perfil_base['cred_superados_anio_1er'] = 0.0

for nota in [5.0, 8.0, 12.0, 14.0]:
    perfil_base['nota_acceso'] = nota
    fila = {col: perfil_base.get(col, 0) for col in cols}
    X = pd.DataFrame([fila], columns=cols)
    X_prep = pipeline.transform(X)
    prob = modelo.predict_proba(X_prep)[0, 1]
    print(f"nota_acceso={nota:.0f} → prob={prob*100:.1f}%")

nota_acceso=5 → prob=84.8%
nota_acceso=8 → prob=78.9%
nota_acceso=12 → prob=65.6%
nota_acceso=14 → prob=69.4%


In [2]:
import joblib, pandas as pd, numpy as np
from pathlib import Path

ROOT = Path(r"C:\Users\mjmor\OneDrive - Universitat Jaume I\2.- AU_UJI")
pipeline = joblib.load(ROOT / "data" / "05_modelado" / "pipeline_preprocesamiento.pkl")
modelo   = joblib.load(ROOT / "data" / "05_modelado" / "models" / "CatBoost__balanced.pkl")

cols = list(pipeline.feature_names_in_)

perfil_base = {
    'nota_acceso':             5.0,
    'situacion_laboral':       11,
    'n_anios_beca':            2,
    'edad_entrada':            19,
    'via_acceso':              10,
    'nota_1er_anio':           np.nan,
    'cred_superados_anio_1er': np.nan,
    'sexo':                    0,
    'tuvo_beca':               1,
    'anios_sin_beca':          2,
    'nota_selectividad':       np.nan,
    'orden_preferencia':       1,
    'anios_gap':               0,
    'universidad_origen':      40,
    'rama':                    3,
    'pais_nombre':             1,
    'provincia':               1,
    'max_pagos':               np.nan,
    'indicador_interrupcion':  0,
}

print("Con NaN en variables no disponibles para prospecto:")
for nota in [5.0, 8.0, 10.0, 12.0, 14.0]:
    perfil_base['nota_acceso'] = nota
    fila = {col: perfil_base.get(col, np.nan) for col in cols}
    X = pd.DataFrame([fila], columns=cols)
    X_prep = pipeline.transform(X)
    prob = modelo.predict_proba(X_prep)[0, 1]
    print(f"  nota={nota:.0f} → {prob*100:.1f}%")

Con NaN en variables no disponibles para prospecto:
  nota=5 → 22.2%
  nota=8 → 25.7%
  nota=10 → 23.7%
  nota=12 → 23.7%
  nota=14 → 23.9%


In [3]:
# ¿El imputer sobreescribe nota_acceso?
num_pipeline = pipeline.named_transformers_['num']
imputer = dict(num_pipeline.steps)['imputer']
print(f"Estrategia imputer: {imputer.strategy}")
print(f"statistics_ (valores de imputación por columna):")
for col, val in zip(cols, imputer.statistics_):
    if col in ['nota_acceso', 'nota_1er_anio', 'cred_superados_anio_1er',
               'n_anios_beca', 'situacion_laboral', 'edad_entrada']:
        print(f"  {col}: {val:.3f}")

Estrategia imputer: median
statistics_ (valores de imputación por columna):
  cred_superados_anio_1er: 54.000
  nota_1er_anio: 6.790
  nota_acceso: 8.220
  edad_entrada: 19.000
  n_anios_beca: 1.000
  situacion_laboral: 8.000
